# 🍌 WallFruits API - Teste Completo

Este notebook demonstra como testar a API WallFruits localmente em `http://127.0.0.1:8000`.

## O que será testado:
- ✅ Requisições GET ao servidor
- ✅ Cadastro de novo usuário (POST)
- ✅ Login e autenticação JWT
- ✅ Criação de ofertas com token
- ✅ Listar e buscar ofertas
- ✅ Favoritar produtos
- ✅ Criar transações
- ✅ Deixar avaliações

**Pré-requisito**: Certifique-se que o servidor está rodando em `http://127.0.0.1:8000`

In [ ]:
import requests
import json
from datetime import datetime

# Configuração da API
BASE_URL = "http://127.0.0.1:8000"

# Armazená dados para usar em próximas requisições
auth_data = {}

print("✅ Bibliotecas importadas com sucesso!")
print(f"🌐 Servidor: {BASE_URL}")

## 1️⃣ GET Request - Testar Servidor

Fazer uma requisição GET simples para verificar se o servidor está respondendo.

In [ ]:
# GET Request - Verificar se o servidor está funcionando
response = requests.get(f"{BASE_URL}/")

print(f"📊 Status Code: {response.status_code}")
print(f"📄 Response:\n")
print(json.dumps(response.json(), indent=2, ensure_ascii=False))

## 2️⃣ POST Request - Registrar Novo Usuário

Criar um novo usuário produtor na plataforma.

In [ ]:
# Dados para registro
user_data = {
    "name": "João Silva",
    "email": "joao@example.com",
    "password": "secret123",
    "role": "producer",  # ou "buyer"
    "phone": "11999999999",
    "location": "São Paulo"
}

# POST Request - Registrar usuário
response = requests.post(
    f"{BASE_URL}/auth/register",
    json=user_data,
    headers={"Content-Type": "application/json"}
)

print(f"📊 Status Code: {response.status_code}")
print(f"📄 Response:\n")
print(json.dumps(response.json(), indent=2, ensure_ascii=False))

# Guardar email para usar no login
auth_data['email'] = user_data['email']
auth_data['password'] = user_data['password']

## 3️⃣ POST Request - Fazer Login

Fazer login e obter um token JWT para autenticação.

In [ ]:
# POST Request - Login
login_data = {
    "email": auth_data['email'],
    "password": auth_data['password']
}

response = requests.post(
    f"{BASE_URL}/auth/login",
    json=login_data,
    headers={"Content-Type": "application/json"}
)

print(f"📊 Status Code: {response.status_code}")
print(f"📄 Response:\n")
result = response.json()
print(json.dumps(result, indent=2, ensure_ascii=False))

# Guardar token para usar em próximas requisições
if response.ok:
    auth_data['token'] = result['access_token']
    print(f"\n✅ Token obtido com sucesso!")
    print(f"🔐 Token: {auth_data['token'][:50]}...")
else:
    print(f"\n❌ Erro no login")

## 4️⃣ POST Request - Criar Oferta de Produto

Criar uma nova oferta usando o token de autenticação.

In [ ]:
# Dados da oferta
offer_data = {
    "product_name": "Banana Prata Premium",
    "description": "Banana de alta qualidade, colhida fresca",
    "category": "frutas",
    "quantity": 50.0,
    "price": 5.50,
    "unit": "kg",
    "location": "São Paulo - SP",
    "latitude": -23.5505,
    "longitude": -46.6333,
    "organic": True,
    "quality_grade": "A",
    "is_negotiable": True,
    "min_order": 5.0
}

# Headers com token de autenticação
headers = {
    "Authorization": f"Bearer {auth_data['token']}",
    "Content-Type": "application/json"
}

# POST Request - Criar oferta
response = requests.post(
    f"{BASE_URL}/offers/",
    json=offer_data,
    headers=headers
)

print(f"📊 Status Code: {response.status_code}")
print(f"📄 Response:\n")
result = response.json()
print(json.dumps(result, indent=2, ensure_ascii=False))

# Guardar offer_id para usar em próximas operações
if response.ok and response.status_code == 201:
    auth_data['offer_id'] = result['id']
    print(f"\n✅ Oferta criada com sucesso!")
    print(f"🏷️ Oferta ID: {auth_data['offer_id']}")
else:
    print(f"\n❌ Erro ao criar oferta")

## 5️⃣ GET Request - Listar Todas as Ofertas

Buscar todas as ofertas com filtros opcionais.

In [ ]:
# GET Request - Listar ofertas com filtros
params = {
    "skip": 0,
    "limit": 10,
    "search": "banana",
    "category": "frutas",
    "min_price": 0,
    "max_price": 100,
    "organic": True,
    "sort_by": "created_at",
    "sort_order": "desc"
}

response = requests.get(
    f"{BASE_URL}/offers/",
    params=params,
    headers={"Authorization": f"Bearer {auth_data.get('token', '')}"}
)

print(f"📊 Status Code: {response.status_code}")
print(f"📄 Response:\n")
result = response.json()

print(f"Total de ofertas: {result['total']}")
print(f"Ofertas retornadas: {len(result['offers'])}")
print(f"\n📊 Estatísticas:")
print(json.dumps(result['stats'], indent=2, ensure_ascii=False))

print(f"\n📋 Primeiras ofertas:")
for i, offer in enumerate(result['offers'][:3], 1):
    print(f"\n{i}. {offer['product_name']}")
    print(f"   Preço: R$ {offer['price']}")
    print(f"   Localização: {offer['location']}")
    print(f"   Visualizações: {offer['views']}")

## 6️⃣ GET Request - Obter Perfil do Usuário

Buscar dados completos do usuário logado.

In [ ]:
# GET Request - Obter perfil do usuário logado
response = requests.get(
    f"{BASE_URL}/auth/me",
    headers={"Authorization": f"Bearer {auth_data['token']}"}
)

print(f"📊 Status Code: {response.status_code}")
print(f"📄 Perfil do Usuário:\n")
result = response.json()
print(json.dumps(result, indent=2, ensure_ascii=False))

## 7️⃣ GET Request - Dashboard do Usuário

Obter dashboard com estatísticas e dados recentes.

In [ ]:
# GET Request - Dashboard
response = requests.get(
    f"{BASE_URL}/dashboard/my-dashboard",
    headers={"Authorization": f"Bearer {auth_data['token']}"}
)

print(f"📊 Status Code: {response.status_code}")
print(f"📊 Dashboard do Usuário:\n")
result = response.json()

# Mostrar estatísticas principais
print("📈 Estatísticas Principais:")
for key, value in result['stats'].items():
    key_formatted = key.replace('_', ' ').title()
    print(f"  • {key_formatted}: {value}")

print(f"\n🎁 Ofertas Recentes:")
for offer in result['recent_offers']:
    print(f"  • {offer['product_name']} - R$ {offer['price']} ({offer['status']})")

print(f"\n📦 Transações Recentes:")
for tx in result['recent_transactions']:
    print(f"  • {tx['product_name']} - R$ {tx['total_price']} ({tx['status']})")

## 📚 Outros Endpoints Disponíveis

Aqui está uma lista de outros endpoints que você pode testar:

### 🛍️ **Ofertas**
- `GET /offers/{offer_id}` - Obter detalhes de uma oferta específica
- `PUT /offers/{offer_id}` - Editar uma oferta
- `DELETE /offers/{offer_id}` - Deletar uma oferta

### ❤️ **Favoritos**
- `POST /favorites/` - Adicionar oferta aos favoritos
- `GET /favorites/my` - Listar seus favoritos
- `DELETE /favorites/{offer_id}` - Remover dos favoritos

### 💰 **Transações**
- `POST /transactions/` - Comprar uma oferta
- `GET /transactions/my` - Listar suas transações
- `PUT /transactions/{transaction_id}` - Atualizar status de transação

### ⭐ **Avaliações**
- `POST /reviews/` - Criar avaliação após compra
- `GET /reviews/user/{user_id}` - Obter avaliações de um usuário
- `GET /reviews/user/{user_id}/stats` - Obter estatísticas de avaliações

### 💬 **Mensagens**
- `POST /messages/` - Enviar mensagem
- `GET /messages/` - Listar mensagens
- `GET /messages/conversations` - Ver conversas
- `WebSocket /messages/ws/{user_id}` - Chat em tempo real

### 📁 **Upload**
- `POST /uploads/profile-image` - Upload de foto de perfil
- `POST /uploads/offer-images` - Upload de fotos da oferta

### 🏷️ **Categorias**
- `GET /categories/` - Listar categorias
- `GET /categories/tree` - Ver categorias em árvore
- `POST /categories/` - Criar categoria (admin)

### 📊 **Analytics**
- `GET /dashboard/insights` - Insights do mercado
- `GET /dashboard/reports/sales` - Relatório de vendas (produtor)

In [ ]:
# Função auxiliar para testar qualquer endpoint
def test_endpoint(method, endpoint, data=None, params=None):
    """
    Função auxiliar para testar endpoints da API
    
    Args:
        method: 'GET', 'POST', 'PUT', 'DELETE'
        endpoint: '/path/to/endpoint'
        data: dados JSON (para POST/PUT)
        params: query parameters
    """
    url = f"{BASE_URL}{endpoint}"
    headers = {"Authorization": f"Bearer {auth_data.get('token', '')}"}
    
    if method == 'GET':
        response = requests.get(url, params=params, headers=headers)
    elif method == 'POST':
        response = requests.post(url, json=data, headers=headers)
    elif method == 'PUT':
        response = requests.put(url, json=data, headers=headers)
    elif method == 'DELETE':
        response = requests.delete(url, headers=headers)
    
    print(f"{'='*60}")
    print(f"🔗 {method} {endpoint}")
    print(f"📊 Status: {response.status_code}")
    print(f"{'='*60}")
    
    if response.text:
        try:
            print(json.dumps(response.json(), indent=2, ensure_ascii=False))
        except:
            print(response.text)
    
    return response

# Exemplo de uso:
# test_endpoint('GET', '/offers/', params={'skip': 0, 'limit': 5})
# test_endpoint('GET', f'/offers/{auth_data["offer_id"]}')
# test_endpoint('POST', '/favorites/', data={'offer_id': auth_data['offer_id']})

print("✅ Função test_endpoint() criada e pronta para usar!")
print("\n💡 Exemplos:")
print("  test_endpoint('GET', '/offers/', params={'skip': 0, 'limit': 5})")
print("  test_endpoint('GET', f'/offers/{auth_data[\"offer_id\"]}')")
print("  test_endpoint('POST', '/favorites/', data={'offer_id': auth_data['offer_id']})")